# 10 • Desempenho, arquivos e limites

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

diario = v.groupby(v['InvoiceDate'].dt.normalize())['valor'].sum()
diario = diario.reindex(pd.date_range(diario.index.min(),diario.index.max(),freq='D'), fill_value=0)
print(diario.head())
# Hipótese: dias não registrados dentro da cobertura tiveram zero venda elegível.

identificados = v.dropna(subset=['CustomerID']).copy()
referencia = v['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = identificados.groupby('CustomerID').agg(
    ultima_compra=('InvoiceDate','max'), frequencia=('InvoiceNo','nunique'), monetario=('valor','sum'))
rfm['recencia_dias'] = (referencia-rfm['ultima_compra'].dt.normalize()).dt.days
print(rfm.head())

### Por que ler em blocos muda a forma de calcular

Imagine um arquivo que não cabe na memória. chunksize=100000 devolve um leitor que entrega até cem mil linhas por vez. O for repete o mesmo preparo para cada bloco. A lista parciais guarda um total por país de cada pedaço; concat reúne esses totais; groupby(level=0).sum() soma países iguais no índice do resultado. Nesse exemplo, level=0 significa o primeiro nível do índice, onde está o país.

A soma das partes é a soma total. Mas a média de um bloco com 10 linhas não pode ter o mesmo peso de outro com mil linhas. Guarde soma e contagem, some ambas e só depois divida. Se estiver calculando vendas elegíveis em blocos, aplique exatamente os mesmos filtros em todos os blocos. O exercício de quantidade bruta usa deliberadamente outra medida, sem os filtros comerciais; seu resultado não é receita elegível.

O argumento index=False ao exportar CSV evita escrever o índice como uma coluna adicional. Uma releitura que preserva shape não garante sozinha que todos os valores e tipos foram preservados; para entregas críticas, compare também chaves, tipos e totais. O exercício usa shape como primeira verificação, não como contrato completo.

### Aprofundamento e cuidados

Leia só o necessário com usecols e defina tipos quando conhecer o contrato. CSV não guarda tipos de forma completa; documente a leitura. Parquet preserva melhor tipos e comprime, mas requer um mecanismo como pyarrow. Excel é útil para pessoas revisarem resultados, mas não deve ser escolhido automaticamente para grandes volumes.

Meça memória com `memory_usage(deep=True)` e tempo com `perf_counter`. `category` pode economizar memória em textos muito repetidos; teste antes, porque não é uma solução universal. Prefira operações vetorizadas e agregações em lote a laços que alteram célula a célula.

`chunksize` lê CSV em blocos. Somar totais por país de cada bloco e somar os parciais funciona; tirar média das médias de blocos pode estar errado. Para média, acumule soma e contagem. Contagens distintas também não são aditivas: o mesmo cliente pode aparecer em vários blocos. Se o volume não cabe em memória ou os cruzamentos ficam caros, considere SQL/DuckDB ou outro mecanismo apropriado e transfira apenas o resultado necessário para pandas.


## Exemplo guiado

In [ ]:
antes = p['species'].memory_usage(deep=True)
categorica = p['species'].astype('category')
print('Memória original:',antes,'bytes; categórica:',categorica.memory_usage(deep=True))

## Exercício 10.1
Leia apenas Country, Quantity e UnitPrice das primeiras cem linhas do CSV da loja.

In [ ]:
# Escreva sua resposta aqui


## Exercício 10.2
Mostre a memória total de r em MiB.

In [ ]:
# Escreva sua resposta aqui


## Exercício 10.3
Compare massa média pela divisão soma/contagem com mean.

In [ ]:
# Escreva sua resposta aqui


## Exercício 10.4
Some quantidades brutas por país em blocos de cem mil linhas. Compare com leitura completa.

In [ ]:
# Escreva sua resposta aqui


## Exercício 10.5
Exporte o resumo por espécie para CSV sem gravar o índice. Releia e confira as dimensões.

In [ ]:
# Escreva sua resposta aqui
